In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Appendix to policy gradient I: the proofs

## Where this fits

- **For your own review**, strongly recommended: the main session states the policy gradient theorem
    - this appendix proves it
- **Position:** Part B, session 1 (the theorem), appendix
- **What is here**
    - **the log-derivative trick** (also called the likelihood-ratio trick): the gradient of an expectation becomes an expectation of a gradient
    - **the derivation of the theorem** (episode-return form): the environment's transition probabilities drop out
    - **why the return-to-go form is equal in expectation**: the score function $\nabla_\theta \log \pi_\theta$ has zero mean
    - **an alternate proof** from per-step rewards
- **Why read it:** the zero-mean fact is used next session (baselines), and the log-derivative form is referred back to later
- **You need:** the objective $J(\theta)$ and the chained episode probability from the main session


## The likelihood ratio trick

**Why this section?** The main session's theorem needs one tool: turning the gradient of an expectation into an expectation of a gradient. It is called the log-derivative trick, or the likelihood-ratio trick.

### Likelihood ratio trick

The likelihood ratio trick states that 
- for a parameterized probability distribution $p_\theta(x)$ 
- and a function $f(x)$:

the gradient of an expectation can be converted into an expectation of $f$ times the gradient of the log-probability (the *score function*).

It is a simple consequence of the Derivative of a Log rule of calculus.
- conditions: $f$ must not depend on $\theta$
- $p_\theta > 0$ where the log is taken
- the gradient may be moved inside the integral (regularity)
- for episodes the integral is a sum over $\tau$


$$
\begin{array}{llll} \\
\nabla_\theta \E_{x \sim p_\theta}[f(x)] & = & \nabla_\theta \int p_\theta(x) f(x) dx  & \text{convert expectation to integral}\\
& = &\int \nabla_\theta p_\theta(x) f(x) dx & \text{move grad inside the integral} \\
& = &\int f(x) \nabla_\theta p_\theta(x)  dx & \text{rearrange term} \\
& = &\int f(x) \left(p_\theta(x) \nabla_\theta \log p_\theta(x) \right)  dx & \text{log trick: } \\
& & & \nabla_\theta p_\theta(x) = p_\theta(x) \nabla_\theta \log p_\theta(x) \\
& = & \E_{x \sim p_\theta} \left[ f(x) \nabla_\theta \log p_\theta(x) \right] & \text{convert integral back to expectation} \\
\end{array}
$$

The "log trick" follows from the rules of calculus
$$
\begin{array}{llll} \\
\nabla_\theta \log p_\theta(x) & = & \frac{1}{p_\theta(x)} * \nabla_\theta p_\theta(x) & \text{Calculus: grad of log, chain rule} \\
\nabla_\theta  p_\theta(x) & = &  p_\theta(x) * \nabla_\theta  \log p_\theta(x) & \text{re-arranging terms} \\
\end{array}
$$

## Computing $\nabla_\theta J(\theta)$

**Why this section?** Putting the trick to work: the derivation of the theorem.

Our first step will be to turn the expectation 
$$
\Exp{\tau \sim \prc{\tau}{\theta}} \,\, { \rlret_{0,\tau} }
$$
into a sum
$$
\sum_\tau \prc{\tau}{\theta} *  { \rlret_{0,\tau} }
$$

where
$$
\prc{\tau}{\theta}
$$
is the probability of trajectory $\tau$

Turning the expectation into a sum and taking the gradient of the expected Episode Return

$$
\begin{array}{llll} \\
\nabla_\theta
\sum_{\tau \sim \prc{\tau}{\theta}} \prc{\tau}{\theta} *  { \rlret_{0,\tau} } 
& = & \sum_{\tau \sim \prc{\tau}{\theta}}{ \nabla_\theta  \, \,\prc{\tau}{\theta} *  { \rlret_{0,\tau} }}  & \text{grad of a sum is sum of the grads} \\ 
& = & \sum_{\tau \sim \prc{\tau}{\theta}}{ (\prc{\tau}{\theta} \nabla_\theta  \log  \prc{\tau}{\theta}) \, \,\rlret_{0,\tau} } & \text{Likelihood ratio trick:} \\
& & & \nabla_\theta \prc{\tau}{\theta} = \prc{\tau}{\theta} * \nabla_\theta \log \prc{\tau}{\theta} \\
\end{array}
$$

We now substitute the chained probability previously derived for 
$$\prc{\tau}{\theta}$$
into the 
$$
\log \prc{\tau}{\theta}
$$

term above.



$$
\begin{array}{llll} \\
\log \prc{\tau}{\theta} & = & \log \left( 
\pr{\stateseq_{\tau,0}}
*
\prod_{\tt=0}^{|\tau|-1} { \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) * \transp(\stateseq_{\tau, \tt+1}, \rewseq_{\tau,\tt+1} | \stateseq_{\tau,\tt}, \actseq_{\tau,\tt})}  \right) & \text{definition of } \prc{\tau}{\theta} \\
& = & 
\log \pr{\stateseq_{\tau,0}}
+
  \sum_{\tt=0}^{|\tau|-1} {
  \left( \,
\log \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) + \log \transp(\stateseq_{\tau, \tt+1}, \rewseq_{\tau,\tt+1} | \stateseq_{\tau,\tt}, \actseq_{\tau,\tt})
\, \right)
} & \text{log of product is sum of logs}  \\
\\
\nabla_\theta \log \prc{\tau}{\theta} & = & 
\sum_{\tt=0}^{|\tau|-1} {
\nabla_\theta  \log \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) 
}  & \text{grad of sum is sum of grads} \\
& & &  \text{Transition Probability terms } \\
& & & \;\;\;\transp( \ldots) \\
& & & \text{are independent of } \theta \\
\end{array}
$$

The initial-state probability $\pr{\stateseq_{\tau,0}}$ does not depend on $\theta$ either, so it also drops out.


To summarize the proof:

$$
\begin{array}{llll} \\
\nabla_\theta \Exp{\tau \sim \pi_\theta} \,\, { \rlret_{0,\tau} } 
& = & \nabla_\theta \sum_\tau \prc{\tau}{\theta} *  { \rlret_{0,\tau} } & \text{turn the expectation into a sum weighted by episode probabilities} \\
& = & \sum_\tau{ \left(\prc{\tau}{\theta} \nabla_\theta  \log  \prc{\tau}{\theta} \right) \, \, *\rlret_{0,\tau} } & \text{Likelihood ratio trick} \\
& = & \sum_\tau{ }{ \left( \prc{\tau}{\theta} * 
    \sum_{\tt=0}^{|\tau|-1} {
\nabla_\theta  \log \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) } \right)
*  { \rlret_{0,\tau} }
} & \text{Transition Probability terms disappear from within the grad} \\
& & & \text{as shown above}\\
& = & \Exp{\tau \sim \prc{\tau}{\theta}}
\left[
\sum_{\tt=0}^{|\tau|-1} {
\nabla_\theta  \log \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt})  }
\; \rlret_{0,\tau} \right]
& \text{turn back into an expectation} \\
& & &\text{involving each step of the episode} \\
\end{array}
$$

### Why substituting the return-to-go $\rlret_{\tau,\tt}$ for the episode return $\rlret_{0,\tau}$ gives the same expectation

Consider the two equivalent forms for expressing the Policy Gradient Theorem

$$
\sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \rlret_{0,\tau}
$$

and

$$
\sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \disc^\tt \rlret_\tt
$$

where:
$$ \rlret_\tt = \sum_{k=\tt}^{|\tau|-1} \disc^{k-\tt} \rewseq_{k+1} $$

and, for the fixed trajectory $\tau$ (the subscript $\tau$ is dropped in this derivation),
$$
\rlret_{0,\tau} = \rlret_0
$$

How can these two forms be mathematically equivalent ?
- since the first form involves $\rlret_0$
    - the rewards *over all steps* of the episode
- and the second form involves $\rlret_\tt$
    - the *future* rewards from step $\tt$ onward
    - and $\rlret_\tt$ and $\rlret_{\tt'}$ for $\tt' > \tt$ include the same rewards
   
Algebraically they appear different.

The answer is that
- the two forms appear *within an expectation*
- which is evaluated over *future* time steps
- so the product of the *past rewards* part of $\rlret_0$ with the score function has expectation $0$ (shown on a later slide)

Here are the details:

- Step 1: Start with the total return $ \rlret_{0,\tau} = \rlret_0 $

Define $L$ to be the expression for the first form of the Theorem:

$
L = \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \rlret_0
= \rlret_0 \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt)
$

---

- Step 2: Expand $ \rlret_0 $ as the sum over rewards

$
\rlret_0 = \sum_{k=0}^{|\tau|-1} \disc^k \rewseq_{k+1}
$

Substitute into $ L $:

$
L = \left(\sum_{k=0}^{|\tau|-1} \disc^k \rewseq_{k+1} \right) \left(\sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt)\right)
$

---

- Step 3: Express as a double sum

$
L = \sum_{\tt=0}^{|\tau|-1} \sum_{k=0}^{|\tau|-1} \disc^k \rewseq_{k+1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt)
$

---

- Step 4: Separate sums over past and future rewards relative to $ \tt $

$
L = \sum_{\tt=0}^{|\tau|-1} \left( \sum_{k=0}^{\tt-1} \disc^k \rewseq_{k+1} + \sum_{k=\tt}^{|\tau|-1} \disc^k \rewseq_{k+1} \right) \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt)
$

---

- Step 5: Rewrite future rewards shifted by $ \tt $

Define $ j = k - \tt $:

$
\sum_{k=\tt}^{|\tau|-1} \disc^k \rewseq_{k+1} = \disc^\tt \sum_{j=0}^{|\tau| - 1 - \tt} \disc^j \rewseq_{\tt+j+1} = \disc^\tt \rlret_\tt
$

---

- Step 6: Substitute back into $ L $

$
L = \sum_{\tt=0}^{|\tau|-1} \left( \sum_{k=0}^{\tt-1} \disc^k \rewseq_{k+1} + \disc^\tt \rlret_\tt \right) \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt)
$

---

**Why the past-rewards term vanishes: the score function has zero mean**

$$
\E_{\actseq \sim \pi_\theta}\bigl[ \nabla_\theta \log \pi_\theta(\actseq \mid \state) \bigr]
= \sum_{\act} \pi_\theta(\act \mid \state) \, \nabla_\theta \log \pi_\theta(\act \mid \state)
= \sum_{\act} \nabla_\theta \pi_\theta(\act \mid \state)
= \nabla_\theta \sum_{\act} \pi_\theta(\act \mid \state)
= \nabla_\theta 1 = 0
$$

- rewards before step $\tt$ are already fixed when $\actseq_\tt$ is chosen, so they multiply a zero-mean term
- the same fact will justify subtracting a baseline (next session)

- for continuous actions the sum over $\act$ is an integral
- the argument is the same

- more carefully: condition on everything up to $\stateseq_\tt$
- the past rewards are then fixed, $\pi_\theta$ depends only on $\stateseq_\tt$, and averaging over $\actseq_\tt$ gives the zero above
- the same argument shows a *baseline* $b(\state)$ that does not depend on the action can be subtracted from the weight without changing the expected gradient


- Step 7: Expectation zeroes out past rewards term

Because rewards before time $\tt$ are fixed before $\actseq_\tt$ is chosen, and the score function has zero mean (previous slide), their expected contribution is zero:

$
\E\left[ \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \sum_{k=0}^{\tt-1} \disc^k \rewseq_{k+1} \right] = 0
$

---

- Step 8: Final form of the policy gradient

Thus,

$
\nabla_\theta J(\theta) = \E_\pi \left[ \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \disc^\tt \rlret_\tt \right]
$

which is the second form of expressing the Policy Gradient Theorem.


### Alternate Proof of the Policy Gradient Theorem (Per-Step Reward Perspective)

Let $ J(\theta) $ be the expected discounted sum of rewards under policy $ \pi_\theta $:

$
J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \disc^{\tt} \rewseq_{\tt+1} \right]
$

---

- Step 1: Expand the Expectation

Rewrite the expectation explicitly:

$
J(\theta) = \sum_{\tau} \prc{\tau}{\theta} \left( \sum_{\tt=0}^{|\tau|-1} \disc^\tt \rewseq_{\tt+1} \right)
$

---

- Step 2: Differentiation w.r.t. $\theta$

$
\nabla_\theta J(\theta) = \sum_{\tau} \nabla_\theta \prc{\tau}{\theta} \left( \sum_{\tt=0}^{|\tau|-1} \disc^\tt \rewseq_{\tt+1} \right)
$

Apply the **likelihood ratio trick**:
$
\nabla_\theta \prc{\tau}{\theta} = \prc{\tau}{\theta} \nabla_\theta \log \prc{\tau}{\theta}
$
So,
$
\nabla_\theta J(\theta) = \sum_{\tau} \prc{\tau}{\theta} \nabla_\theta \log \prc{\tau}{\theta} \left( \sum_{\tt=0}^{|\tau|-1} \disc^\tt \rewseq_{\tt+1} \right)
$
Or equivalently,
$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \nabla_\theta \log \prc{\tau}{\theta} \left( \sum_{\tt=0}^{|\tau|-1} \disc^\tt \rewseq_{\tt+1} \right) \right]
$

---

- Step 3: Break Down $\log \prc{\tau}{\theta}$

Recall,
$
\log \prc{\tau}{\theta} = \sum_{\tt=0}^{|\tau|-1} \log \pi_\theta(\actseq_\tt \vert \stateseq_\tt) + \text{terms independent of } \theta
$
So,
$
\nabla_\theta \log \prc{\tau}{\theta} = \sum_{\tt'=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tt'}|\stateseq_{\tt'})
$
Substitute:

$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt'=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tt'}|\stateseq_{\tt'}) \cdot \left( \sum_{\tt=0}^{|\tau|-1} \disc^\tt \rewseq_{\tt+1} \right) \right]
$

---

- Step 4: Swap Order of Summation

$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt'=0}^{|\tau|-1} \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tt'}|\stateseq_{\tt'}) \disc^\tt \rewseq_{\tt+1} \right]
$

Switch the order:

$
= \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \sum_{\tt'=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tt'} | \stateseq_{\tt'}) \disc^\tt \rewseq_{\tt+1} \right]
$

---

-  Step 5: Analyze the causal relationship

An action cannot affect earlier rewards (causality): for $\tt < \tt'$ the reward $\rewseq_{\tt+1}$ is fixed before $\actseq_{\tt'}$ is chosen, so by the zero-mean score function fact (an earlier slide) the expected product is zero. This proof therefore uses the same fact as the first.

Thus, the only contributing terms are where $\tt' \leq \tt$:

$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt'=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tt'} | \stateseq_{\tt'}) \left( \sum_{\tt=\tt'}^{|\tau|-1} \disc^\tt \rewseq_{\tt+1} \right) \right]
$

---

- Step 6: Recognize the return-to-go term

$
\sum_{\tt=\tt'}^{|\tau|-1} \disc^\tt \rewseq_{\tt+1} = \disc^{\tt'} \sum_{j=0}^{|\tau|-1-\tt'} \disc^j \rewseq_{\tt'+j+1} = \disc^{\tt'} \rlret_{\tt'}
$

So, we may write:

$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \cdot \disc^\tt \rlret_\tt \right]
$
Many references and implementations drop $\disc^\tt$ (it is $1$ when $\disc = 1$). The exact gradient of the discounted objective keeps it.

---

- Step 7: Final form (policy gradient theorem)

$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_\tt | \stateseq_\tt) \, \disc^\tt \rlret_\tt \right]
$

---

**Conclusion:**  
By starting with the expectation of per-step rewards and applying the likelihood ratio trick, we arrive at the same policy gradient theorem:  
each action's gradient is weighted by the discounted return-to-go from that time, $\disc^\tt \rlret_\tt$ (not just the immediate reward).



# What we established, and what it sets up

- **Established**
    - $\nabla_\theta p_\theta(x) = p_\theta(x) \nabla_\theta \log p_\theta(x)$: the log-derivative trick
    - the log of an episode's probability is a sum of logs, and the transition probabilities do not depend on $\theta$, so $\nabla_\theta \log \prc{\tau}{\theta}$ is a sum of $\nabla_\theta \log \pi_\theta$ terms
    - hence the episode-return form of the theorem, with no $\transp$
    - the score function has zero mean, so rewards already received do not change the expected gradient: the return-to-go form
- **Sets up**
    - the next session subtracts a *baseline* from the weight: it keeps the expected gradient because the score function has zero mean
    - later sessions refer back to the log-derivative form of the gradient
